# Import

In [2]:
from __future__ import annotations

import argparse
import time
from pathlib import Path

import numpy as np
from PIL.ImageOps import grayscale

from dataset_loader import DEFAULT_DATASET_ROOT, load_labeled_image_dataset, stratified_split
from rust_bridge import MLPRust, OVRLinearClassifier, TaskMode

# Fonction

In [3]:
#---Parser pour utiliser le code avec des commandes---
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Train the Rust linear baseline and MLP on the game screenshot dataset."
    )
    parser.add_argument(
        "--root",
        type=Path,
        default=DEFAULT_DATASET_ROOT,
        help="Root directory containing FPS, METROIDVANIA and MOBA subfolders.",
    )
    parser.add_argument("--width", type=int, default=8, help="Resized image width.")
    parser.add_argument("--height", type=int, default=6, help="Resized image height.")
    parser.add_argument("--grayscale", dest="grayscale", action="store_true", help="Use grayscale images.")
    parser.add_argument("--rgb", dest="grayscale", action="store_false", help="Use RGB images.")
    parser.add_argument("--test-ratio", type=float, default=0.2, help="Fraction used for test split.")
    parser.add_argument("--seed", type=int, default=42, help="Random seed for the split.")
    parser.add_argument("--linear-lr", type=float, default=0.01, help="Learning rate for the linear baseline.")
    parser.add_argument(
        "--linear-steps",
        "--linear-epochs",
        dest="linear_steps",
        type=int,
        default=20_000,
        help="Random training steps for the linear baseline.",
    )
    parser.add_argument("--mlp-lr", type=float, default=0.01, help="Learning rate for the MLP.")
    parser.add_argument(
        "--mlp-steps",
        "--mlp-epochs",
        dest="mlp_steps",
        type=int,
        default=50_000,
        help="Random training steps for the naive MLP.",
    )
    parser.add_argument(
        "--mlp-layers",
        type=int,
        nargs="*",
        default=[16, 8],
        help="Hidden layer sizes for the MLP. Example: --mlp-layers 16 8",
    )
    parser.set_defaults(grayscale=True)
    return parser.parse_args()

#---Affichage des résultats---
def _print_results(
    title: str,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    class_names: tuple[str, ...],
) -> None:
    accuracy = float(np.mean(np.all(y_true == y_pred, axis=1)))
    confusion = _confusion_matrix(y_true, y_pred, len(class_names))

    # display of one result block
    print(f"{title}:")
    print(f"Accuracy: {accuracy:.3f}")
    print("Matrice de confusion (lignes=reel, colonnes=predit) :")
    print("       " + " ".join(f"{name[:5]:>6}" for name in class_names))
    for row_index, row in enumerate(confusion):
        print(f"{class_names[row_index][:5]:>5} " + " ".join(f"{value:>6}" for value in row))

#---Matrice de confusion---
def _confusion_matrix(y_true: np.ndarray,
                      y_pred: np.ndarray,
                      class_count: int
                      ) -> np.ndarray:
    matrix = np.zeros((class_count, class_count), dtype=np.int32)
    true_indices = np.argmax(y_true, axis=1)
    pred_indices = np.argmax(y_pred, axis=1)

    for true_index, pred_index in zip(true_indices, pred_indices):
        matrix[true_index, pred_index] += 1

    return matrix


# Code

In [4]:
etat_grayscale = True #True ou false, activation de la nuance de gris
chemin = r"C:\Users\theot\Pictures\Dataset"
width = int(40 * 16/9)
height = 40

#---Création du Dataset transformer---
bundle = load_labeled_image_dataset(root=chemin, image_size=(width, height), grayscale=etat_grayscale)

#--Séparation des données en test et train---
split_bundle = stratified_split(bundle, test_ratio=0.2, seed=42)

#---Affichage des informations---
print("=== Dataset ===")
print(f"Chemin: {chemin}")
print(f"Classes: {', '.join(bundle.class_names)}")
print(f"Nb images par classe: {bundle.counts_by_class}")
color_mode = "grayscale" if etat_grayscale else "rgb"
print(f"Format apres pretraitement: {width}x{height} {color_mode}")
print(f"Nb features: {bundle.x.shape[1]}")
print(f"Nb images train: {split_bundle.x_train.shape[0]}")
print(f"Nb images test: {split_bundle.x_test.shape[0]}")
print(f"Images ignorees: {len(bundle.skipped_paths)}")
if bundle.skipped_paths:
    for skipped_path in bundle.skipped_paths[:5]:
        print(f"  - {skipped_path.name}")
print()

=== Dataset ===
Chemin: C:\Users\theot\Pictures\Dataset
Classes: FPS, METROIDVANIA, MOBA
Nb images par classe: {'FPS': 597, 'METROIDVANIA': 743, 'MOBA': 606}
Format apres pretraitement: 71x40 grayscale
Nb features: 2840
Nb images train: 1555
Nb images test: 391
Images ignorees: 0



# Entrainement des modèles

In [5]:
#---Entrainement modèle linéaire---
start_time = time.perf_counter()
epochs = 100000             #Nombre de fois où le modèle s'entrainera sur l'ensemble des données
pas_apprentissage = 0.001    #Pas d'apprentissage du modèle linéaire

linear = OVRLinearClassifier(
        input_dim=split_bundle.x_train.shape[1],
        output_dim=len(bundle.class_names),
        learning_rate=pas_apprentissage,
    )
linear.fit(split_bundle.x_train, split_bundle.y_train, epochs=epochs)
linear_predictions = linear.predict_labels(split_bundle.x_test)
linear.close()

#---Résultats---
print("=== Resultats ===")
_print_results("Linear", split_bundle.y_test, linear_predictions, bundle.class_names)
print()
print(f"Temps total: {time.perf_counter() - start_time:.2f}s")


=== Resultats ===
Linear:
Accuracy: 0.928
Matrice de confusion (lignes=reel, colonnes=predit) :
          FPS  METRO   MOBA
  FPS    100     11      9
METRO      1    146      2
 MOBA      2      3    117

Temps total: 1.59s


In [6]:
#---Entrainement modèle MLP---
start_time = time.perf_counter()


mlp = MLPRust(
        layer_sizes=[split_bundle.x_train.shape[1],
                     #*args.mlp_layers,
                     len(bundle.class_names)],
        #learning_rate=,
        task_mode=TaskMode.CLASSIFICATION,
    )
mlp.fit(split_bundle.x_train, split_bundle.y_train,
        #steps=args.mlp_steps
        )
mlp_predictions = mlp.predict_labels(split_bundle.x_test)
mlp.close()

#---Résultats---
_print_results("MLP", split_bundle.y_test, mlp_predictions, bundle.class_names)
print()
print(f"Temps total: {time.perf_counter() - start_time:.2f}s")

MLP:
Accuracy: 0.629
Matrice de confusion (lignes=reel, colonnes=predit) :
          FPS  METRO   MOBA
  FPS    106     13      1
METRO      9    140      0
 MOBA    106     16      0

Temps total: 3.31s


# Visualisation des données

In [ ]:
score = np.stack()